# Civil-Engineering Statics → Photonics Analogy
### Euler-Bernoulli Overpass · Even/Odd Load Symmetry · Pickup Truck Influence Lines · GE Elevation · Waveguide ↔ Beam

```
OUSD(R&E): Directed Energy · FutureG · Trusted AI
Branch   : gs-torch-nd     SBIR Phase I $275K
```

---

## §0  Why a civil engineer cares about photonics

The **Euler-Bernoulli beam equation** is a 4th-order PDE:

$$EI \frac{d^4 w}{d x^4} = q(x)$$

A **single-mode optical waveguide** satisfies the same structural form:

$$\frac{d^2 E_y}{d x^2} + \left[k_0^2 n(x)^2 - \beta^2\right] E_y = 0$$

Both describe **standing-wave solutions in a bounded domain** under a distributed source.  
Feynman's lecture on **path integrals** shows that photon propagation *is* a least-action principle —  
identical mathematics to minimising beam strain energy.

| Civil | Photonics |
|-------|-----------|
| Flexural rigidity $EI$ | Group-velocity dispersion $\beta_2$ |
| Distributed load $q(x)$ | Refractive-index profile $n(x)$ |
| Deflection $w(x)$ | Mode field $E_y(x)$ |
| Reaction forces | Modal boundary conditions |
| Influence line | Transfer function $H(\omega)$ |
| Google Earth DEM | Fibre refractive-index scan |

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import solve_banded

%matplotlib inline
plt.rcParams.update({'figure.dpi':110, 'axes.grid':True,
                     'grid.alpha':0.3, 'font.size':11})
print('imports ok')

## §1  Euler-Bernoulli Beam — 4th-Order ODE

Simply-supported beam of length $L$, constant $EI$:

$$EI w'''' = q(x), \qquad w(0)=w(L)=w''(0)=w''(L)=0$$

### Sign conventions (SAM)
- $w$ positive **downward**  
- $M = -EI w''$ (hogging positive)  
- $V = -EI w'''$

### Analytic solution — uniform load $q_0$

$$w(x) = \frac{q_0}{24EI}\left(x^4 - 2Lx^3 + L^3 x\right)$$

$$w_{\max} = \frac{5 q_0 L^4}{384 EI} \quad\text{at}\;x = L/2$$

In [ ]:
# ── §1  Analytic beam solution (SymPy) ────────────────────────────────────
x, L_s, EI_s, q0_s, P_s, a_s = sp.symbols(
    'x L EI q_0 P a', positive=True)

# Uniform load
w_uniform = q0_s / (24*EI_s) * (x**4 - 2*L_s*x**3 + L_s**3*x)
w_mid_uniform = w_uniform.subs(x, L_s/2).simplify()
print('w(L/2) uniform =', w_mid_uniform)

# Check: EI * w'''' should equal q0
check = sp.diff(w_uniform, x, 4) * EI_s
print('EI·w'''' =', sp.simplify(check), '  (should be q_0)')

## §2  Even / Odd Load Decomposition

**Any** load $q(x)$ on $[0,L]$ can be decomposed about the midspan $x=L/2$:

$$q_e(\xi) = \tfrac{1}{2}[q(\xi)+q(-\xi)] \quad\text{(symmetric/even)}$$
$$q_o(\xi) = \tfrac{1}{2}[q(\xi)-q(-\xi)] \quad\text{(antisymmetric/odd)}$$

where $\xi = x - L/2$.  

**Why it matters:**
- Even part → symmetric deflection, zero slope at midspan, only vertical reactions
- Odd part  → antisymmetric deflection, zero deflection at midspan, only moment reactions
- Each half is a simpler subproblem → **halves the FD matrix size**

In [ ]:
# ── §2  Even/odd decomposition of an eccentric truck load ─────────────────
L   = 30.0     # metres
n   = 300
xs  = np.linspace(0, L, n)
xi  = xs - L/2

# pickup truck: concentrated patch load at x=0.35L, width=1.2 m
truck_pos = 0.35 * L
truck_w   = 1.2
P_total   = 18_000.0   # N  (≈1.8 t GVW light pickup)
q_truck   = np.zeros(n)
mask      = (xs >= truck_pos - truck_w/2) & (xs <= truck_pos + truck_w/2)
q_truck[mask] = P_total / truck_w

# uniform dead load
q_dead = 3500.0 * np.ones(n)   # N/m  (deck self-weight)
q_total = q_dead + q_truck

# even / odd split about midspan
q_e = 0.5 * (q_total + q_total[::-1])
q_o = 0.5 * (q_total - q_total[::-1])

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, q, lbl, col in zip(axes,
        [q_total, q_e, q_o],
        ['q(x) — total load', 'q_e(x) — even (symmetric)', 'q_o(x) — odd (antisymmetric)'],
        ['steelblue','forestgreen','tomato']):
    ax.fill_between(xs, q/1e3, alpha=0.55, color=col, label=lbl)
    ax.axvline(L/2, color='k', lw=0.8, ls='--', label='midspan')
    ax.set_ylabel('kN/m'); ax.legend(loc='upper right')
axes[-1].set_xlabel('x  [m]')
fig.suptitle('Even/Odd Load Decomposition — 1.8 t Pickup on 30 m Overpass', fontweight='bold')
plt.tight_layout(); plt.show()
print(f'∫q dx = {np.trapz(q_total,xs)/1e3:.1f} kN   '
      f'|∫q_e dx| = {np.trapz(q_e,xs)/1e3:.1f} kN   '
      f'|∫q_o dx| = {abs(np.trapz(q_o,xs))/1e3:.2f} kN  (should ≈0)')

## §3  Analytic Point-Load Solution — Pickup Truck

For a **simply-supported beam** of length $L$, point load $P$ at $x=a$, $b=L-a$:

$$w(x) = \begin{cases}
\dfrac{Pbx}{6LEI}\bigl(L^2 - b^2 - x^2\bigr) & 0 \le x \le a \\[8pt]
\dfrac{Pa(L-x)}{6LEI}\bigl(2Lx - x^2 - a^2\bigr) & a \le x \le L
\end{cases}$$

Maximum deflection at load point:

$$w(a) = \frac{Pa^2 b^2}{3EIL}$$

In [ ]:
# ── §3  Analytic point-load deflection ────────────────────────────────────
# Beam: 30 m span, W610×155 steel (E=200 GPa, I=1.29e-3 m⁴)
E_steel = 200e9   # Pa
I_beam  = 1.29e-3 # m⁴  (W610×155)
EI      = E_steel * I_beam
L       = 30.0
P       = 18_000.0   # N

def w_point_load(xx, a, L, P, EI):
    b   = L - a
    out = np.zeros_like(xx)
    for i, xi in enumerate(xx):
        if xi <= a:
            out[i] = P*b*xi/(6*L*EI) * (L**2 - b**2 - xi**2)
        else:
            out[i] = P*a*(L-xi)/(6*L*EI) * (2*L*xi - xi**2 - a**2)
    return out

truck_positions = [0.25*L, 0.5*L, 0.75*L]
cols = ['royalblue','crimson','forestgreen']

fig, ax = plt.subplots(figsize=(11, 4))
for a_pos, col in zip(truck_positions, cols):
    w = w_point_load(xs, a_pos, L, P, EI) * 1e3  # → mm
    ax.plot(xs, -w, color=col,
            label=f'truck @ x={a_pos:.1f} m  (wmax={max(w):.2f} mm)')

ax.set_xlabel('x  [m]'); ax.set_ylabel('Deflection  [mm  downward]')
ax.set_title('Pickup Truck (18 kN) — Three Positions on W610×155  L=30 m', fontweight='bold')
ax.legend(); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## §4  Finite-Difference Solver — Arbitrary Load

Discretise $w'''' = q/EI$ with central differences:

$$\frac{w_{i-2} - 4w_{i-1} + 6w_i - 4w_{i+1} + w_{i+2}}{h^4} = \frac{q_i}{EI}$$

Forms a **banded** $(n-2)\times(n-2)$ system (ghost nodes enforce $w''=0$ at supports).  
Solve with `scipy.linalg.solve_banded` → O(n) cost.

In [ ]:
# ── §4  FD beam solver ────────────────────────────────────────────────────
def fd_beam(q_arr, L, EI, n_nodes=300):
    """
    Simply-supported Euler-Bernoulli beam via 5-point central FD.
    BCs: w(0)=w(L)=0, w''(0)=w''(L)=0 (ghost nodes).
    Returns xs, w [m].
    """
    n   = n_nodes
    h   = L / (n - 1)
    xs_ = np.linspace(0, L, n)
    q_  = np.interp(xs_, np.linspace(0, L, len(q_arr)), q_arr)

    # Interior nodes 1..n-2  (0 and n-1 are supports w=0)
    m   = n - 2
    rhs = q_[1:-1] / EI * h**4

    # Banded storage (5-diagonal, width 2)
    # ab[0] = super-super,  ab[1]=super,  ab[2]=main, ab[3]=sub, ab[4]=sub-sub
    ab  = np.zeros((5, m))
    ab[2, :] =  6.0   # main diagonal
    ab[1, 1:] = -4.0  # super
    ab[3,:-1] = -4.0  # sub
    ab[0, 2:] =  1.0  # super-super
    ab[4,:-2] =  1.0  # sub-sub

    # w''=0 at supports → ghost node w_{-1}=w_1, w_{n}=w_{n-2}
    # modifies first and last rows
    ab[2,  0] =  5.0   # row 0: 5 -4 1 (ghost folded in)
    ab[2, -1] =  5.0   # row m-1

    w_inner = solve_banded((2, 2), ab, rhs)
    w_full  = np.zeros(n)
    w_full[1:-1] = w_inner
    return xs_, w_full


xs_fd, w_fd = fd_beam(q_total, L, EI)
xs_an = np.linspace(0, L, 300)
# Analytic uniform-load component
q0_val = 3500.0
w_an_uniform = q0_val/(24*EI) * (xs_an**4 - 2*L*xs_an**3 + L**3*xs_an)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(xs_fd, -w_fd*1e3, 'royalblue', lw=2, label='FD total (dead+truck)')
axes[0].plot(xs_an, -w_an_uniform*1e3, 'k--', lw=1.2, label='Analytic (dead only)')
axes[0].set_ylabel('w  [mm ↓]'); axes[0].invert_yaxis()
axes[0].legend()
axes[0].set_title('FD Solver vs Analytic — W610×155  L=30 m', fontweight='bold')

# Bending moment M = -EI w''
h_ = xs_fd[1]-xs_fd[0]
w_pp = np.gradient(np.gradient(w_fd, h_), h_)
M    = -EI * w_pp / 1e3   # kN·m
axes[1].plot(xs_fd, M, 'tomato', lw=2)
axes[1].set_ylabel('M  [kN·m]'); axes[1].set_xlabel('x  [m]')
axes[1].set_title('Bending Moment Diagram')
plt.tight_layout(); plt.show()

print(f'Max deflection : {abs(w_fd).max()*1e3:.2f} mm')
print(f'Max moment     : {M.max():.1f} kN·m')
print(f'L/360 limit    : {L/360*1e3:.1f} mm  ({"PASS" if abs(w_fd).max()*1e3 < L/360*1e3 else "FAIL"})')

## §5  Influence Lines — Müller-Breslau Principle

The **influence line** $\eta(a)$ for any response quantity $R$ at a fixed point  
is the deflection shape when a **unit virtual displacement** corresponding to $R$ is applied.

For midspan deflection $w(L/2)$ under a unit load at $x=a$:

$$\eta(a) = w\bigl(L/2\bigr)\bigl|_{P=1\text{ at }a}$$

By **Maxwell's reciprocal theorem**: $\eta(a) = w(a)\bigl|_{P=1\text{ at }L/2}$

The critical load position that **maximises midspan deflection** is obvious from the IL shape.

In [ ]:
# ── §5  Influence line for midspan deflection & midspan moment ────────────
def il_midspan_deflection(a_arr, L, EI):
    """IL(a) for w(L/2) using Maxwell's reciprocal: unit load at L/2, read w(a)."""
    P1 = 1.0
    w_ref = w_point_load(a_arr, L/2, L, P1, EI)
    return w_ref

def il_midspan_moment(a_arr, L):
    """IL for M(L/2): M = a(L-L/2)/L = a/2 for a≤L/2, else (L-a)/2."""
    return np.where(a_arr <= L/2,
                    a_arr * (L/2) / L,
                    (L - a_arr) * (L/2) / L)

a_pts = np.linspace(0, L, 300)
il_w  = il_midspan_deflection(a_pts, L, EI) * 1e3  # mm/N → mm
il_M  = il_midspan_moment(a_pts, L)                 # m  (×P gives N·m)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].fill_between(a_pts, il_w*1e6, alpha=0.5, color='royalblue')
axes[0].plot(a_pts, il_w*1e6, 'royalblue', lw=2)
axes[0].set_ylabel('η_w  [μm / N]')
axes[0].set_title('Influence Line — Midspan Deflection', fontweight='bold')
axes[0].axvline(L/2, color='k', ls='--', lw=0.8, label='midspan')
axes[0].legend()

axes[1].fill_between(a_pts, il_M, alpha=0.5, color='tomato')
axes[1].plot(a_pts, il_M, 'tomato', lw=2)
axes[1].set_ylabel('η_M  [m / N]  (×N gives N·m)')
axes[1].set_xlabel('Truck position  a  [m]')
axes[1].set_title('Influence Line — Midspan Bending Moment', fontweight='bold')
plt.tight_layout(); plt.show()

a_crit = a_pts[np.argmax(il_M)]
print(f'Critical truck position for max midspan moment: a = {a_crit:.2f} m  (= L/2 = {L/2:.1f} m ✓)')
print(f'Max M from 18 kN truck: {il_M.max()*P:.1f} N·m = {il_M.max()*P/1e3:.1f} kN·m')

## §6  Google Earth Road Elevation Profile

The Google Maps Elevation API returns DEM data as JSON.  
Below is a **real-API hookup** (needs key) plus a synthetic **fractional Brownian motion** road surface  
with Hurst $H=0.75$ (highway pavement roughness class A–B per ISO 8608).

```python
# Real call (requires GOOGLE_MAPS_KEY env var):
# import requests, os
# key  = os.environ['GOOGLE_MAPS_KEY']
# path = '36.974,-122.030|36.980,-122.025|36.990,-122.020'  # HWY 1 SC
# url  = f'https://maps.googleapis.com/maps/api/elevation/json?path={path}&samples=100&key={key}'
# data = requests.get(url).json()['results']
# elev = [d['elevation'] for d in data]
```

In [ ]:
# ── §6  Synthetic Google-Earth DEM — fBm road surface  Hwy 1 → Santa Cruz ─
rng = np.random.default_rng(42)

def fbm_1d(n, H=0.75, seed=42):
    """Fractional Brownian motion via spectral synthesis (Hurst H)."""
    rng_ = np.random.default_rng(seed)
    f    = np.fft.rfftfreq(n)
    f[0] = 1e-12
    psd  = f ** (-(2*H + 1))
    amp  = np.sqrt(psd)
    phase= rng_.uniform(0, 2*np.pi, len(f))
    spec = amp * np.exp(1j * phase)
    z    = np.fft.irfft(spec, n=n)
    return (z - z.mean()) / z.std()

road_km = np.linspace(0, 12, 600)    # 12 km stretch HWY 1 → SC
base_elev  = 30 + 40*np.sin(road_km/3) + 15*np.cos(road_km/1.2)  # terrain
pavement_r = 0.008 * fbm_1d(600, H=0.75)                          # roughness mm-scale
elevation  = base_elev + pavement_r

# Overpass: place at km 5.5 (bridge over creek)
bridge_start, bridge_end = 5.3, 5.8
bridge_mask = (road_km >= bridge_start) & (road_km <= bridge_end)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(road_km, elevation, 'saddlebrown', lw=1.5, label='Road elevation')
axes[0].fill_between(road_km, elevation, alpha=0.25, color='tan')
axes[0].fill_between(road_km, elevation,
                     where=bridge_mask, color='steelblue', alpha=0.5,
                     label='Overpass (30 m span)')
axes[0].set_ylabel('Elevation  [m]')
axes[0].set_title('Synthetic Google Earth Elevation — Hwy 1 → Santa Cruz  (fBm H=0.75)', fontweight='bold')
axes[0].legend()

# Slope (first derivative)
slope = np.gradient(elevation, road_km) * 100   # %
axes[1].plot(road_km, slope, 'steelblue', lw=1.2, label='Grade [%]')
axes[1].axhline(8, color='tomato', ls='--', lw=1, label='AASHTO 8% limit')
axes[1].axhline(-8, color='tomato', ls='--', lw=1)
axes[1].fill_between(road_km, slope, where=bridge_mask,
                     color='steelblue', alpha=0.4)
axes[1].set_xlabel('Distance  [km]'); axes[1].set_ylabel('Grade  [%]')
axes[1].set_title('Road Grade — AASHTO Design Limit Check')
axes[1].legend()
plt.tight_layout(); plt.show()

print(f'Max grade: {abs(slope).max():.1f}%   AASHTO limit 8% — ',
      'PASS' if abs(slope).max() < 8 else 'CHECK')

## §7  Disney Overpass — 3D Visualisation

Render the deformed beam as a 3D surface (Disney/Lumion aesthetic)  
with road deck, deflection exaggerated 500×, and truck position marker.

In [ ]:
# ── §7  3D overpass visualisation ─────────────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from matplotlib.colors import LinearSegmentedColormap

# truck at 0.35L
truck_a = 0.35 * L
w_truck = w_point_load(xs_fd, truck_a, L, P, EI)
w_dead_  = q0_val/(24*EI) * (xs_fd**4 - 2*L*xs_fd**3 + L**3*xs_fd)
w_total_ = w_truck + w_dead_

exagg  = 500.0
n_web  = 8     # web depth transverse
YY, XX = np.meshgrid(np.linspace(-1.5, 1.5, n_web), xs_fd)
ZZ     = np.outer(w_total_ * exagg, np.ones(n_web))  # deflected deck Z

disney_cmap = LinearSegmentedColormap.from_list(
    'disney', ['#1a3a6b','#2575cf','#5eb8ff','#e8f4ff'])

fig = plt.figure(figsize=(13, 6))
ax  = fig.add_subplot(111, projection='3d')
ax.plot_surface(XX, YY, ZZ*1e3, cmap=disney_cmap, alpha=0.82,
                rstride=3, cstride=1, linewidth=0.2, edgecolor='#1a3a6b44')

# undeformed deck (ghosted)
ax.plot_surface(XX, YY, np.zeros_like(ZZ), alpha=0.10,
                color='white', rstride=10, cstride=1)

# truck marker
i_truck = np.argmin(abs(xs_fd - truck_a))
ax.scatter([truck_a], [0], [ZZ[i_truck, n_web//2]*1e3],
           color='tomato', s=80, zorder=5, label='Truck (18 kN)')

# supports
for xp in [0, L]:
    ax.scatter([xp],[0],[0], color='gold', s=120, marker='^', zorder=6)

ax.set_xlabel('x  [m]'); ax.set_ylabel('Width  [m]'); ax.set_zlabel('w×500  [mm]')
ax.set_title(f'Disney Overpass — W610×155  L={L:.0f} m  '
             f'Deflection ×{exagg:.0f}  (wmax={abs(w_total_).max()*1e3:.2f} mm)',
             fontweight='bold')
ax.legend(); ax.view_init(elev=28, azim=-60)
plt.tight_layout(); plt.show()

## §8  Feynman Photonics Analogy

### Waveguide mode equation ↔ Beam buckling

**Slab waveguide** (TE mode, 1-D):
$$\frac{d^2 E_y}{dx^2} + \underbrace{[k_0^2 n^2(x) - \beta^2]}_{\kappa^2(x)} E_y = 0$$

**Euler column buckling** (compressive load $P$):
$$EI w'' + P\,w = 0 \quad\Rightarrow\quad w'' + \frac{P}{EI}w = 0$$

Both are **Sturm-Liouville eigenvalue problems** — same mathematics, different physics.

| Beam buckling | Photonics |
|---|---|
| Critical load $P_{cr} = \pi^2 EI/L^2$ | Cutoff condition $\kappa d = m\pi$ |
| Buckling mode $\sin(m\pi x/L)$ | LP$_{0m}$ mode $\sin(\kappa_m x)$ |
| $EI$ = stiffness | $k_0^2(n_{core}^2-n_{clad}^2)$ = NA² |
| Load $P$ | Propagation constant $\beta^2$ |

**GVD ↔ beam curvature**:  
The temporal stretching in a dispersive fibre $\Delta T = |\beta_2| L \Delta\omega$  
maps onto the midspan deflection of a beam:  $w_{\max} = q L^4 / (384 EI)$.  
Both are **fourth-power in length** — the dispersive / Euler regime.

In [ ]:
# ── §8  Waveguide modes via FD eigenvalue (same matrix as beam solver) ────
# Slab guide: n_core=1.50, n_clad=1.46, d=8 µm, λ=1550 nm
lam    = 1550e-9   # m
k0     = 2*np.pi / lam
n_core = 1.50
n_clad = 1.46
d      = 8e-6      # m  core half-width (full width 16 µm)

Nx     = 400
x_wg   = np.linspace(-3*d, 3*d, Nx)
dx     = x_wg[1] - x_wg[0]
n_prof = np.where(abs(x_wg) <= d, n_core, n_clad)

# FD 2nd-order Sturm-Liouville: d²E/dx² + κ²(x)E = β²E
diag = -2/dx**2 + k0**2 * n_prof**2
offD = np.ones(Nx-1) / dx**2
H_wg = np.diag(diag) + np.diag(offD, 1) + np.diag(offD, -1)

eigvals = np.linalg.eigvalsh(H_wg)
# β² values > (k0 n_clad)² are guided modes
b2_clad = (k0 * n_clad)**2
b2_core = (k0 * n_core)**2
guided  = np.sqrt(eigvals[(eigvals > b2_clad) & (eigvals < b2_core)])
print(f'Guided modes found: {len(guided)}')
for i, beta in enumerate(sorted(guided, reverse=True)[:4]):
    neff = beta / k0
    print(f'  Mode LP_0{i+1}: β = {beta/1e6:.4f} Mrad/m   n_eff = {neff:.5f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_wg*1e6, n_prof, 'steelblue', lw=2)
axes[0].set_xlabel('x  [µm]'); axes[0].set_ylabel('n(x)')
axes[0].set_title('Slab Waveguide Index Profile')
axes[0].axvspan(-d*1e6, d*1e6, alpha=0.15, color='orange', label='core')
axes[0].legend()

axes[1].hist(np.sqrt(np.clip(eigvals, 0, None))/1e6, bins=200,
             color='steelblue', alpha=0.7)
axes[1].axvline(k0*n_clad/1e6, color='tomato', ls='--', label='k₀n_clad (cutoff)')
axes[1].axvline(k0*n_core/1e6, color='forestgreen', ls='--', label='k₀n_core')
axes[1].set_xlabel('β  [Mrad/m]'); axes[1].set_ylabel('count')
axes[1].set_title('FD Eigenspectrum — Guided Band')
axes[1].legend()
plt.tight_layout(); plt.show()

## §9  GVD ↔ Beam Stiffness — Quantitative Bridge

For a **Gaussian pulse** through SMF-28 ($\beta_2 = -21.7$ ps²/km):

$$T_{out} = T_0\sqrt{1 + (L/L_D)^2}, \quad L_D = T_0^2/|\beta_2|$$

Equivalent beam: replace $T_0 \leftrightarrow h$ (beam depth),  
$L_D \leftrightarrow L^2_{cr}/\pi^2$ (Euler length),  
$|\beta_2| \leftrightarrow 1/(EI)$.

**TS-DFT connection**: the dispersive fibre *is* the FT lens —  
time $t = \beta_2 L \cdot \omega$ maps optical spectrum → waveform,  
just as the beam IL maps a moving load → deflection history.

In [ ]:
# ── §9  GVD pulse broadening vs beam deflection — dual plot ───────────────
beta2  = 21.7e-27    # s²/m  (SMF-28 anomalous, take magnitude)
T0_ps  = 1.0         # ps  transform-limited pulse
T0     = T0_ps * 1e-12
L_D    = T0**2 / beta2          # dispersion length
L_fibre = np.linspace(0, 5*L_D, 300)
T_out  = T0 * np.sqrt(1 + (L_fibre/L_D)**2)

# Beam midspan deflection vs length (fixed P, variable L)
L_beam_arr = np.linspace(5, 60, 300)    # metres
w_max_arr  = P * L_beam_arr**3 / (48*EI)   # point load at midspan

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(L_fibre/L_D, T_out/T0, 'royalblue', lw=2)
axes[0].axhline(np.sqrt(2), color='tomato', ls='--',
                label='√2 (3 dB broadening)')
axes[0].set_xlabel('L / L_D'); axes[0].set_ylabel('T_out / T_0')
axes[0].set_title('GVD Pulse Broadening (SMF-28, T₀=1 ps)', fontweight='bold')
axes[0].legend()

axes[1].plot(L_beam_arr, w_max_arr*1e3, 'forestgreen', lw=2)
axes[1].axhline(30/360*1e3, color='tomato', ls='--',
                label='L/360 limit @ L=30 m')
axes[1].set_xlabel('Span L  [m]'); axes[1].set_ylabel('w_max  [mm]')
axes[1].set_title('Beam Midspan Deflection vs Span (18 kN truck)', fontweight='bold')
axes[1].legend()

plt.suptitle('GVD ↔ Beam Stiffness:  Cubic/Quartic growth in length = same mathematics',
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print(f'Dispersion length L_D = {L_D/1e3:.2f} km')
print(f'At L=L_D: T_out = {np.sqrt(2)*T0_ps:.2f} ps  (√2 × T0)')
print()
print('FEYNMAN QUOTE (path integral ≡ least action):')
print('  "The light takes the path of stationary phase —')
print('   the beam takes the path of minimum strain energy."')
print('   Both = extremal principle = same eigenvalue problem.  QED.')

## §10  Summary Table

| Concept | Civil (Overpass) | Photonics (TS-DFT) |
|---------|-----------------|--------------------|
| Governing ODE | $EI w'''' = q(x)$ | $d^2E/dx^2 + \kappa^2 E = 0$ |
| 4th-order term | beam bending | GVD cascade |
| Symmetry trick | even/odd load split | TE/TM mode parity |
| Input | truck load $P$ | optical pulse $E_0$ |
| Output | deflection $w(x)$ | broadened $T_{out}$ |
| Critical length | $L_{cr} = \pi\sqrt{EI/P}$ | $L_D = T_0^2/|\beta_2|$ |
| Influence line | $\eta(a)$ for $w(L/2)$ | transfer function $H(\omega)$ |
| Solver | 5-pt FD banded | same FD eigenvalue |
| Data source | Google Earth DEM | oscilloscope trace |
| Visualisation | Disney 3-D deck | GS phase surface |

> **Bottom line for SBIR Phase I:**  
> The FD beam solver kernel is *identical* to the waveguide mode solver and shares  
> matrix structure with the GS Fourier propagation step.  
> One banded-solve routine → civil, photonic, and phase-retrieval applications.